In [8]:
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

In [9]:
text_dataset = tf.data.TextLineDataset("ind.txt")
type(text_dataset)

tensorflow.python.data.ops.readers.TextLineDatasetV2

In [10]:
# Hyperparameters
VOCAB_SIZE = 10000
ENGLISH_SEQUENCE_LENGTH = 16
INDONESIA_SEQUENCE_LENGTH = 16
EMBEDDING_DIM = 256
BATCH_SIZE = 32

In [7]:
# Vectorization Layer
english_vectorization_layer = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    max_tokens = VOCAB_SIZE,
    output_mode = 'int',
    output_sequence_length = ENGLISH_SEQUENCE_LENGTH
)

indonesian_vectorization_layer = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    max_tokens = VOCAB_SIZE,
    output_mode = 'int',
    output_sequence_length = INDONESIA_SEQUENCE_LENGTH
)

NameError: name 'tf' is not defined

In [5]:
# Preprocess Text
def split_text(text):
    text = tf.strings.split(text, '\t')
    input_1 = text[:1]
    input_2 = 'starttoken ' + text[1:2] + ' endtoken'
    return input_1, input_2

def vectorize(text):
    text = tf.strings.split(text, '\t')
    input_1 = text[:1]
    start_input = 'starttoken ' + text[1:2]
    end_input = text[1:2] + ' endtoken'
    print(f"Vectorization -- Start Input: {start_input} End Input: {end_input}")
    return {
        'input_1': english_vectorization_layer(input_1),
        'input_2': indonesian_vectorization_layer(start_input)
    }, indonesian_vectorization_layer(end_input)

In [6]:
# Preprocessing
splitted_dataset = text_dataset.map(split_text)

# Create training data
print("Creating english training data and vectorization layer...")
english_training_data = splitted_dataset.map(lambda x, y: x)
english_vectorization_layer.adapt(english_training_data)

print("Creating indonesian training data and vectorization layer...")
indonesian_training_data = splitted_dataset.map(lambda x, y: y)
indonesian_vectorization_layer.adapt(indonesian_training_data)

full = text_dataset.shuffle(20000, seed=42, reshuffle_each_iteration=False)
n_val = 1559
val_raw, train_raw = full.take(n_val), full.skip(n_val)

train_dataset = (train_raw.map(vectorize).shuffle(4096).unbatch().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset   = (val_raw.map(vectorize).unbatch().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

Creating english training data and vectorization layer...
Creating indonesian training data and vectorization layer...
Vectorization -- Start Input: Tensor("add:0", shape=(None,), dtype=string) End Input: Tensor("add_1:0", shape=(None,), dtype=string)
Vectorization -- Start Input: Tensor("add:0", shape=(None,), dtype=string) End Input: Tensor("add_1:0", shape=(None,), dtype=string)


In [7]:
len_indonesia_vocab = len(indonesian_vectorization_layer.get_vocabulary())
index_to_word={x:y for x, y in zip(range(len_indonesia_vocab), indonesian_vectorization_layer.get_vocabulary())}
print(index_to_word)

{0: '', 1: '[UNK]', 2: 'starttoken', 3: 'endtoken', 4: 'tom', 5: 'aku', 6: 'tidak', 7: 'yang', 8: 'di', 9: 'saya', 10: 'kamu', 11: 'itu', 12: 'ini', 13: 'dia', 14: 'apa', 15: 'akan', 16: 'ada', 17: 'bisa', 18: 'dengan', 19: 'ke', 20: 'kami', 21: 'kita', 22: 'tahu', 23: 'apakah', 24: 'ingin', 25: 'mary', 26: 'untuk', 27: 'pergi', 28: 'kau', 29: 'adalah', 30: 'sudah', 31: 'suka', 32: 'anda', 33: 'harus', 34: 'dan', 35: 'punya', 36: 'lebih', 37: 'sedang', 38: 'orang', 39: 'dari', 40: 'mereka', 41: 'sangat', 42: 'makan', 43: 'mana', 44: 'sini', 45: 'dalam', 46: 'hari', 47: 'banyak', 48: 'pada', 49: 'semua', 50: 'pernah', 51: 'jangan', 52: 'lagi', 53: 'rumah', 54: 'melakukan', 55: 'siapa', 56: 'seorang', 57: 'bahasa', 58: 'mau', 59: 'bukan', 60: 'melihat', 61: 'masih', 62: 'berapa', 63: 'telah', 64: 'tahun', 65: 'belum', 66: 'hal', 67: 'saja', 68: 'buku', 69: 'tolong', 70: 'tiga', 71: 'tinggal', 72: 'kalau', 73: 'datang', 74: 'baik', 75: 'waktu', 76: 'sebuah', 77: 'mobil', 78: 'sekarang', 7

In [8]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True)
    def call(self, x):
        x = self.embedding(x)
        return self.lstm(x)

In [17]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.w_1 = tf.keras.layers.Dense(units)
        self.w_2 = tf.keras.layers.Dense(units)
        self.w_output = tf.keras.layers.Dense(1)
    
    def call(self, previous_decoder, encoder_state):
        scores = self.w_output(
            tf.tanh(
                self.w_1(tf.expand_dims(previous_decoder, -2))
                +
                self.w_2(encoder_state)
            )
        )
        attention_weight = tf.nn.softmax(scores, axis=1)
        context_vector = attention_weight * encoder_state
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weight    

In [18]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_units, sequence_length, decoder_units):
        super(Decoder, self).__init__()
        self.attention = BahdanauAttention(decoder_units)
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_units)
        self.gru = tf.keras.layers.GRU(decoder_units, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(vocab_size, activation="softmax")
        self.sequence_length = sequence_length
        self.decoder_units = decoder_units
        
    def call(self, encoder_output, shifted_output):
        outputs = []
        attention_weight = []
        shifted_output = self.embedding(shifted_output)
        hidden_state = tf.zeros((tf.shape(encoder_output)[0], self.decoder_units))
        
        for t in range(0, self.sequence_length):
            context_vector, attention_weight = self.attention(hidden_state, encoder_output)
            dec_input = context_vector + shifted_output[:, t]
            output, hidden_state = self.gru(tf.expand_dims(dec_input, 1), initial_state=hidden_state)
            outputs.append(output[:, 0])
            
        outputs = tf.convert_to_tensor(outputs)
        outputs = tf.transpose(outputs, perm=[1,0,2])
        outputs = self.dense(outputs)
        
        return outputs, attention_weight

In [20]:
# Model Creation
HIDDEN_UNITS=256
EMBEDDING_DIM=256

### ENCODER
input = tf.keras.layers.Input(shape=(ENGLISH_SEQUENCE_LENGTH, ), dtype="int64", name="input_1")
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS)
encoder_output = encoder(input)

### DECODER
shifted_output = tf.keras.layers.Input(shape=(INDONESIA_SEQUENCE_LENGTH, ), dtype="int64", name="input_2")
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, INDONESIA_SEQUENCE_LENGTH, HIDDEN_UNITS)
decoder_output, attention_weight = decoder(encoder_output, shifted_output)

bahdanau = tf.keras.Model([input, shifted_output], decoder_output)
bahdanau.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 16)]         0           []                               
                                                                                                  
 encoder_4 (Encoder)            (None, 16, 256)      3085312     ['input_1[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 16)]         0           []                               
                                                                                                  
 decoder_3 (Decoder)            ((None, 16, 10000),  5656593     ['encoder_4[0][0]',              
                                 (None, 16, 1))                   'input_2[0][0]']            

In [21]:
bahdanau.compile(
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"], 
    optimizer="adam"
)

In [22]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", 
    patience=5, 
    restore_best_weights=True
)

In [25]:
history = bahdanau.fit(
    train_dataset, 
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stopping],
    batch_size=BATCH_SIZE
)

Epoch 1/10
439/439 [==============================] - 39s 40ms/step - loss: 2.5101 - accuracy: 0.6636 - val_loss: 2.1172 - val_accuracy: 0.7066
Epoch 2/10
439/439 [==============================] - 14s 32ms/step - loss: 2.0351 - accuracy: 0.7075 - val_loss: 2.0485 - val_accuracy: 0.7123
Epoch 3/10
439/439 [==============================] - 14s 32ms/step - loss: 1.9498 - accuracy: 0.7113 - val_loss: 1.9830 - val_accuracy: 0.7142
Epoch 4/10
439/439 [==============================] - 14s 32ms/step - loss: 1.8671 - accuracy: 0.7167 - val_loss: 1.9110 - val_accuracy: 0.7193
Epoch 5/10
439/439 [==============================] - 14s 31ms/step - loss: 1.7635 - accuracy: 0.7255 - val_loss: 1.8448 - val_accuracy: 0.7303
Epoch 6/10
439/439 [==============================] - 14s 32ms/step - loss: 1.6438 - accuracy: 0.7414 - val_loss: 1.7433 - val_accuracy: 0.7448
Epoch 7/10
439/439 [==============================] - 14s 31ms/step - loss: 1.5218 - accuracy: 0.7551 - val_loss: 1.6587 - val_accuracy:

In [4]:
def translate(sentences):
    tokenize = english_vectorization_layer([sentences])
    shifted_output = 'starttoken'
    result = ''
    
    for i in range(INDONESIA_SEQUENCE_LENGTH):
        tokenize_shifted = indonesian_vectorization_layer([shifted_output])
        output = bahdanau.predict([tokenize, tokenize_shifted])
        # ubah balik token translate ke index kata
        word_index = tf.argmax(output, axis=-1)[0][i].numpy() 
        
        current_word = index_to_word[word_index]
        if current_word == "endtoken":
            break
        
        shifted_output += ' ' + current_word
        
        # nambah spasi ke semua loop kecuali yg pertama
        if i != 0:
            result += ' '
        result += current_word

    return result

In [6]:
translate("Prabowo is a palm president")

NameError: name 'english_vectorization_layer' is not defined